# Sales Value Classification
**MLOps Project — Google Colab Phase**

Dataset: Data Penjualan (`data_penjualan.csv`)
Target: `High_Value` — **High Value (1)** / **Low Value (0)** berdasarkan total dari harga dan order penjulan

---
Pipeline Colab:
1. Setup & Install
2. Load Dataset
3. Exploratory Data Analysis (EDA)
4. Preprocessing
5. Baseline Modelling
6. Hyperparameter Tuning
7. Export & Download

## 1. Setup & Install Dependencies

In [ ]:
!pip install pandas scikit-learn matplotlib seaborn joblib -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print('Libraries loaded ✓')

## 2. Load Dataset

> **Pilih salah satu cara upload dataset:**

In [ ]:
# Option A: Upload langsung dari komputer
from google.colab import files
uploaded = files.upload()  # upload data_penjualan.csv
DATA_PATH = 'data_penjualan.csv'

# Option B: Mount Google Drive (uncomment jika pakai Drive)
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = '/content/drive/MyDrive/MLOps/data_penjualan.csv'

In [ ]:
df = pd.read_csv(DATA_PATH, sep=";")
print(f'Shape     : {df.shape}')
print(f'Columns   : {list(df.columns)}')
print(f'\nPreview:')
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
print('=== Dataset Info ===')
df.info()
print('\n=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0])

In [ ]:
print('=== Total Statistics ===')
print(df['Total'].describe())
median_val = df['Total'].median()
print(f'\nMedian (Threshold untuk High_Value): {median_val:.2f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['Total'].dropna().hist(bins=40, ax=axes[0], color='#3B82F6', edgecolor='white')
axes[0].axvline(median_val, color='navy', linestyle='--', linewidth=2,
                label=f'Median = {median_val:.1f}%')
axes[0].set_title('Distribusi Total Penjualan', fontsize=13)
axes[0].set_xlabel('Total (%)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

df.groupby('Tanggal')['Total'].mean().plot(
    ax=axes[1], marker='o', color='#10B981', linewidth=2)
axes[1].set_title('Rata-rata Total Penjualan per Tanggal', fontsize=13)
axes[1].set_ylabel('Avg Total (%)')
axes[1].set_xlabel('Year')
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('eda_distribusi_tren.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# Per Jenis Produk
df.groupby('Jenis Produk')['Total'].mean().sort_values().plot(
    kind='barh', ax=ax, color='#3B82F6')
ax.set_title('Rata-rata Total Penjualan per Jenis Produk', fontsize=12)
ax.set_xlabel('Avg Total Penjualan')

plt.tight_layout()
plt.savefig('eda_jenis_produk.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# Korelasi antara Harga dan Jumlah Order
ax.scatter(df['Harga'], df['Jumlah Order'], alpha=0.5, color='#EF4444')
ax.set_title('Hubungan Harga dan Jumlah Order', fontsize=12)
ax.set_xlabel('Harga')
ax.set_ylabel('Jumlah Order')

plt.tight_layout()
plt.savefig('eda_korelasi.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Distribusi Fitur Lain
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df['Jumlah Order'].hist(bins=30, ax=axes[0], color='#6366F1', edgecolor='white')
axes[0].set_title('Distribusi Jumlah Order')

df['Harga'].hist(bins=30, ax=axes[1], color='#EC4899', edgecolor='white')
axes[1].set_title('Distribusi Harga')

plt.tight_layout()
plt.show()

## 4. Preprocessing

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
import pandas as pd # Ensure pandas is imported for pd.to_numeric

def preprocess(df):
    df = df.copy()

    # Step 1: Handle Date column
    if "Tanggal" in df.columns:
        df["Tanggal"] = pd.to_datetime(df["Tanggal"], format="%d/%m/%Y")
        df["Year"] = df["Tanggal"].dt.year
        df["Month"] = df["Tanggal"].dt.month
        df["Day"] = df["Tanggal"].dt.day
        df.drop(columns=["Tanggal"], inplace=True)
        print(f'[INFO] Setelah mapping tanggal    : {df.shape}')

    # Step 2: Handle missing values
    df.dropna(inplace=True)
    print(f'[INFO] Setelah handle missing   : {df.shape}')

    # Step 3: Buat target High_Value dari Total
    if "Total" in df.columns:
        median_val = df["Total"].median()
        print(f'[INFO] Threshold (median)       : {median_val:.2f}')
        df["High_Value"] = (df["Total"] > median_val).astype(int)
        df.drop(columns=["Total"], inplace=True)

    # Step 4: Encode kolom kategorik
    cat_cols = ["Jenis Produk"]
    le = LabelEncoder()
    for col in cat_cols:
        if col in df.columns:
            df[col] = le.fit_transform(df[col].astype(str))

    # Step 5: Scale kolom numerik
    num_cols = ["Jumlah Order", "Harga", "Year", "Month", "Day"]
    scaler = StandardScaler()
    existing_num_cols = [c for c in num_cols if c in df.columns]
    if existing_num_cols:
        df[existing_num_cols] = scaler.fit_transform(df[existing_num_cols])

    print(f'[INFO] Preprocessing selesai    : {df.shape}')
    return df

df_processed = preprocess(df)
print('\n=== High_Value Distribution ===')
print(df_processed['High_Value'].value_counts())
print(df_processed['High_Value'].value_counts(normalize=True).map('{:.1%}'.format))
df_processed.head()

In [ ]:
os.makedirs('data_penjualan_preprocessed', exist_ok=True)
df_processed.to_csv('data_penjualan_preprocessed/preprocessed.csv', index=False)
print('✓ Saved: data_penjualan_preprocessed/preprocessed.csv')
print(f'  Shape   : {df_processed.shape}')
print(f'  Columns : {list(df_processed.columns)}')

## 5. Baseline Modelling

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay
)

X = df_processed.drop(columns=['High_Value'])
y = df_processed['High_Value']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size : {X_train.shape}')
print(f'Test size  : {X_test.shape}')

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print('=== Classification Report (Baseline) ===')
print(classification_report(y_test, y_pred, target_names=['Low Value', 'High Value']))
print(f'AUC-ROC : {roc_auc_score(y_test, y_proba):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Low Value', 'High Value']).plot(
    ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix — Baseline', fontsize=13)

RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1], color='#E8553E')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.4)
axes[1].set_title('ROC Curve — Baseline', fontsize=13)

plt.tight_layout()
plt.savefig('baseline_eval.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
importances.plot(kind='barh', ax=ax, color='#8B5CF6', edgecolor='white')
ax.set_title('Feature Importances — Baseline Model', fontsize=13)
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators':      [50, 100, 200],
    'max_depth':         [5, 10, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf':  [1, 2]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

print('Running GridSearchCV... (estimasi 2-5 menit)')
grid_search.fit(X_train, y_train)

print(f'\n✓ Best Params  : {grid_search.best_params_}')
print(f'✓ Best F1 (CV) : {grid_search.best_score_:.4f}')

In [ ]:
best_model = grid_search.best_estimator_
y_pred_tuned  = best_model.predict(X_test)
y_proba_tuned = best_model.predict_proba(X_test)[:, 1]

print('=== Classification Report (Tuned) ===')
print(classification_report(y_test, y_pred_tuned, target_names=['Low Value', 'High Value']))
print(f'AUC-ROC : {roc_auc_score(y_test, y_proba_tuned):.4f}')

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

metrics = {
    'Accuracy':  [accuracy_score(y_test, y_pred),  accuracy_score(y_test, y_pred_tuned)],
    'F1 Score':  [f1_score(y_test, y_pred),         f1_score(y_test, y_pred_tuned)],
    'Precision': [precision_score(y_test, y_pred),  precision_score(y_test, y_pred_tuned)],
    'Recall':    [recall_score(y_test, y_pred),     recall_score(y_test, y_pred_tuned)],
    'AUC-ROC':   [roc_auc_score(y_test, y_proba),  roc_auc_score(y_test, y_proba_tuned)],
}

df_compare = pd.DataFrame(metrics, index=['Baseline', 'Tuned']).T
print(df_compare.to_string())

fig, ax = plt.subplots(figsize=(10, 4))
df_compare.plot(kind='bar', ax=ax, color=['#6366F1', '#F59E0B'], edgecolor='white', width=0.6)
ax.set_ylim(0, 1.1)
ax.set_title('Perbandingan Baseline vs Tuned Model', fontsize=13)
ax.set_ylabel('Score')
ax.legend(['Baseline', 'Tuned'])
ax.tick_params(axis='x', rotation=0)
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', padding=3, fontsize=9)
plt.tight_layout()
plt.savefig('comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Export & Download

In [ ]:
import joblib

joblib.dump(model,      'rf_model.pkl')
joblib.dump(best_model, 'rf_model_tuned.pkl')
df_compare.to_csv('model_metrics.csv')

print('✓ rf_model.pkl')
print('✓ rf_model_tuned.pkl')
print('✓ model_metrics.csv')
print('✓ data_penjualan_preprocessed/preprocessed.csv')

In [ ]:
from google.colab import files

for fname in [
    'data_penjualan_preprocessed/preprocessed.csv',
    'rf_model.pkl',
    'rf_model_tuned.pkl',
    'model_metrics.csv',
    'eda_distribusi_tren.png',
    'eda_jenis_produk.png',
    'eda_korelasi.png',
    'baseline_eval.png',
    'feature_importance.png',
    'comparison.png',
]:
    if os.path.exists(fname):
        files.download(fname)
        print(f'✓ Downloaded: {fname}')
    else:
        print(f'✗ Not found : {fname}')